# Network Analysis — Streckenveränderungen 2023–2025

Das Zürcher Tramnetz ist kein statisches Objekt.
Im Analysezeitraum 2023–2025 gab es mit dem Fahrplanwechsel Dezember 2023 den größten Netzausbau in der Geschichte der VBZ.
Dieses Notebook untersucht **was** sich verändert hat, **wo** und **wann** — und was das für die Pünktlichkeit bedeutet.

**Zentrale Fragen:**
1. Wo haben sich die meisten Änderungen abgespielt? — Haltestellen und Stadtteile
2. Wieviel hat sich verändert? — Quantifizierung pro Linie und gesamt
3. Wann fanden die Änderungen statt? — Zeitachse
4. Hat sich die Lage nach dem Ausbau verbessert oder verschlechtert? — Einlaufzeit neuer Abschnitte
5. Welche Knotenpunkte sind kritische Hotspots? — Kaskaden und Linienüberschneidungen
6. Welche Stadtteile profitieren? — Versorgungsqualität

**Rückschluss für alle weiteren Analysen:** → am Ende dieses Notebooks

## Setup

In [ ]:
from zh_tram_flow.notebook import *
import zh_tram_flow.analytics.network as an

TRAIN, TEST, lf = setup_analysis("03_analysis_2-network")
lf_all   = pl.concat([pl.scan_parquet(TRAIN), pl.scan_parquet(TEST)])
lf_delay = lf_all.filter(pl.col("canceled") == False)
lf_clean = (
    lf_all
    .filter(pl.col("canceled") == False)
    .filter(~((pl.col("operating_date").dt.year() == 2025) & (pl.col("operating_date").dt.month() >= 11)))
    .filter(pl.col("line_name") != "E")
    .filter(pl.col("stop_sequence") > 1)
)

%load_ext autoreload
%autoreload 2

In [ ]:
# ── GTFS-Vergleichsdaten laden ────────────────────────────────────────────────
# Quelle: sf_data-research GTFS j23 / j24 / j25
# Stop-Sequenzen wurden in der Netzstruktur-Analyse extrahiert und gecacht.
# Falls noch nicht vorhanden: einmalig aus sf_data-research neu berechnen.

import pandas as pd
from pathlib import Path

SF_GTFS = PATHS["root"].parent / "sf_data-research" / "data" / "raw" / "vbz" / "gtfs"

YEARS = {
    "j23": SF_GTFS / "2023_google_transit",
    "j24": SF_GTFS / "2024_google_transit",
    "j25": SF_GTFS / "2025_google_transit",
}

def get_stops_per_line(year, path):
    """Gibt pro Linie die repräsentativen Haltestellen-Namen zurück (direction 0)."""
    routes = pd.read_csv(path / "routes.txt", dtype=str)
    trips  = pd.read_csv(path / "trips.txt",  dtype=str)
    stops  = pd.read_csv(path / "stops.txt",  dtype=str)
    stops["stop_lat"] = stops["stop_lat"].astype(float)
    stops["stop_lon"] = stops["stop_lon"].astype(float)

    tram_r = routes[
        routes["route_id"].str.startswith("1-") &
        routes["route_short_name"].str.match(r"^\d+$|^E$")
    ][["route_id","route_short_name"]]

    trips_t = (trips.merge(tram_r, on="route_id")
               [lambda df: df["direction_id"] == "0"]
               [["route_short_name","shape_id","trip_id"]])

    rep = (trips_t.groupby(["route_short_name","shape_id"])
           .size().reset_index(name="n")
           .sort_values("n", ascending=False)
           .groupby("route_short_name").first().reset_index())

    result = {}
    trip_ids = [trips_t[trips_t["shape_id"] == sid]["trip_id"].iloc[0]
                for sid in rep["shape_id"].tolist()]

    st_df = (pl.scan_csv(str(path / "stop_times.txt"), infer_schema_length=100)
             .filter(pl.col("trip_id").is_in(trip_ids))
             .sort(["trip_id","stop_sequence"]).collect())

    for _, r in rep.iterrows():
        ln  = r["route_short_name"]
        sid = r["shape_id"]
        mask = trips_t["shape_id"] == sid
        if not mask.any(): continue
        tid = trips_t[mask]["trip_id"].iloc[0]
        stop_ids = st_df.filter(pl.col("trip_id") == tid)["stop_id"].to_list()
        st_m = stops[stops["stop_id"].isin(stop_ids)].copy()
        names = set(st_m["stop_name"].dropna())
        lats  = st_m.set_index("stop_name")["stop_lat"].to_dict()
        lons  = st_m.set_index("stop_name")["stop_lon"].to_dict()
        result[ln] = {"names": names, "coords": {n: (lats.get(n), lons.get(n)) for n in names}}
    return result

log("Lade GTFS j23 / j24 / j25 ...")
gtfs = {yr: get_stops_per_line(yr, path) for yr, path in YEARS.items()}
all_lines = sorted(set(ln for yr in gtfs for ln in gtfs[yr]), key=lambda x: int(x) if x.isdigit() else 99)
success(f"{len(all_lines)} Linien geladen: {all_lines}")

In [ ]:
# ── Änderungsmatrix aufbauen ──────────────────────────────────────────────────
LINE_COLORS = {
    "2":"#E20A16","3":"#00892F","4":"#11296F","5":"#734522","6":"#CA7D3C",
    "7":"#000000","8":"#8AB51F","9":"#11296F","10":"#E12472","11":"#00892F",
    "12":"#92D6E3","13":"#FFCC00","14":"#008DC5","15":"#E20A16","17":"#8E224D",
    "18":"#E20A16","19":"#E20A16","E":"#E20A16",
}

rows = []
for ln in all_lines:
    n23 = gtfs["j23"].get(ln, {}).get("names", set())
    n24 = gtfs["j24"].get(ln, {}).get("names", set())
    n25 = gtfs["j25"].get(ln, {}).get("names", set())
    added_j24    = n24 - n23
    removed_j24  = n23 - n24
    added_j25    = n25 - n24
    removed_j25  = n24 - n25
    rows.append({
        "line": ln, "n_j23": len(n23), "n_j24": len(n24), "n_j25": len(n25),
        "added_j24": len(added_j24), "removed_j24": len(removed_j24),
        "added_j25":  len(added_j25),  "removed_j25":  len(removed_j25),
        "changed_j24": bool(added_j24 or removed_j24),
        "changed_j25": bool(added_j25 or removed_j25),
        "names_j23": n23, "names_j24": n24, "names_j25": n25,
        "names_added_j24": added_j24, "names_removed_j24": removed_j24,
        "names_added_j25": added_j25,  "names_removed_j25":  removed_j25,
        "coords_j23": gtfs["j23"].get(ln, {}).get("coords", {}),
        "coords_j24": gtfs["j24"].get(ln, {}).get("coords", {}),
        "coords_j25": gtfs["j25"].get(ln, {}).get("coords", {}),
    })
changes = pd.DataFrame(rows)
success(f"Änderungsmatrix: {len(changes)} Linien")
show_df(changes[["line","n_j23","n_j24","n_j25","added_j24","removed_j24","added_j25","removed_j25"]].set_index("line"))

## Überblick — Das Netz im Wandel

In [ ]:
an.plot_new_stops_by_district(changes, lf_all, cfg)
show_df(an.table_new_stops_by_district(changes, lf_all))

Im Fahrplanwechsel Dezember 2023 (j23 → j24) wurden **{changed_lines_j24} von {n_lines} Linien** verändert.
Der Ausbau ist einseitig: fast alle Änderungen passierten in diesem einen Wechsel.
j24 → j25 ist vergleichsweise stabil.

> Die interaktive Karte mit allen Linien, Haltestellen und Jahresvergleich:
> [`reports/figures/tram_lines_map.html`](../reports/figures/tram_lines_map.html)

### Interaktive Karte

In [ ]:
from pathlib import Path
map_path = Path("maps/network_changes.html")
map_path.parent.mkdir(exist_ok=True)
an.create_network_changes_map(changes, map_path)
from IPython.display import IFrame
display(IFrame(str(map_path), width=900, height=500))

**Karte direkt öffnen:** [`../reports/figures/tram_lines_map.html`](../reports/figures/tram_lines_map.html)  
*(Falls die IFrame-Vorschau oben nicht geladen hat — Link im Browser öffnen)*

## Wo? — Räumliche Verteilung der Änderungen

In [ ]:
section_header("Neue Haltestellen nach Stadtkreis")
an.plot_new_stops_by_district(changes, lf_all, cfg)
show_df(an.table_new_stops_by_district(changes, lf_all))

**Karte direkt öffnen:** [`../reports/figures/network_changes_map.html`](../reports/figures/network_changes_map.html)  
*(Neue Halte ab Dez 2023 in Orange · Entfernte Halte in Grau)*

In [ ]:
log("Netto-Änderungen pro Linie (j23→j24 und j24→j25)")

In [ ]:
show_df(an.table_network_netto_changes(changes))

**Beobachtung:** Das Balkendiagramm zeigt einen überraschenden Befund: Nicht die erwarteten peripheren Aussenkreise (Schwamendingen für L9, Auzelg für L11, Altstetten/Albisrieden für L13) dominieren, sondern die Innenstadt-Kreise.

**Top-Kreise nach neuen Haltestellen (ab Dez 2023):**
| Stadtkreis | Neue Halte |
|:---|---:|
| Kreis 1 (Altstadt/City) | **12** |
| Kreis 3 (Wiedikon) | 10 |
| Kreis 6 (Unterstrass) | 10 |
| Kreis 7 (Fluntern/Witikon) | 9 |
| Kreis 2 (Enge/Wollishofen) | 6 |
| Kreis 5 (Industriequartier) | 5 |
| Kreise 4, 11 | je 4 |

Kreis 1 erhielt mit 12 neuen Haltestellen am meisten — die Innenstadt. Das widerspricht der ursprünglichen Annahme, dass die GTFS-Differenzen bei L13 (+19 Halte) und L11 (+13) peripheren Streckenausbau abbilden. Die Innenstadt-Dominanz deutet darauf hin, dass die GTFS-Änderungen weniger neue Aussenäste kodieren als Umroutings, veränderte Wendeäste oder neue Kursführungen durch die Innenstadtachsen. Das deckt sich mit dem externen Recherche-Hinweis (F-NET-01), dass keine formalen Tramstrecken-Umbauten dokumentiert sind.

## Wieviel? — Quantifizierung der Änderungen

In [ ]:
an.plot_network_stop_count_by_line(changes, cfg)

In [ ]:
show_df(an.table_network_netto_changes(changes))

**Beobachtung:** Das Quantifizierungs-Chart und die Tabelle liefern klare Zahlen zum Ausmass der GTFS-Änderungen.

**Netto-Änderungen j23 → j24 (die grossen Bewegungen):**
| Linie | j23 | j24 | j25 | Δ j23→j24 | Anmerkung |
|:---|---:|---:|---:|---:|:---|
| **L13** | 11 | 30 | 30 | **+19** | Grösste Änderung (+173%) |
| **L11** | 20 | 33 | 34 | **+13** | +16 hinzu, -3 entfernt |
| **L9** | 24 | 32 | 32 | **+8** | +13 hinzu, -5 entfernt |
| L18 | 0 | 16 | 0 | +16 | Temporäre Linie — nur 2024 aktiv |
| **L6** | 24 | 16 | 16 | **−8** | Netto 8 Halte entfernt |

**Anomalie L2:** j23=31 Halte → j24=21 (−10) → j25=31 (+10 zurück). Die Haltestellen-Zahl schwankt stark zwischen den Jahren, obwohl L2 als "stabil" gilt. Wahrscheinlich eine GTFS-Routing-Variante (andere Wendeäste oder Kursführung je Fahrplan), keine reale Streckenkürzung und Rückkehr.

**Neu in j25 (nicht j24):** L5 wächst von 9 auf 14 Halte (+5) — der einzige nennenswerte Ausbau beim j24→j25-Übergang.

**Stabil über alle Jahre:** L10, L12, L14, L17 (identische Haltestellen-Zahl in j23/j24/j25) — echte Referenzlinien für jahresübergreifende Vergleiche.

## Wann? — Zeitachse der Änderungen

In [ ]:
an.plot_monthly_delay_all_lines(lf_all, cfg)

In [ ]:
show_df(an.table_delay_before_after_switch(lf_all))

**Beobachtung:** Kein erkennbarer Knick oder Sprung bei Jan 2024 (Fahrplanwechsel). Die Zeitreihen für alle Linien verlaufen kontinuierlich über den Wechsel hinweg.

**Veränderte vs. stabile Linien — Δ vor/nach Fahrplanwechsel:**
| Linie | Typ | vor (2023) | nach (2024–25) | Δ |
|:---|:---|---:|---:|---:|
| L11 | ✦ verändert | 65.1s | 70.4s | **+5.3s** |
| L13 | ✦ verändert | 51.6s | 53.0s | +1.4s |
| L9 | ✦ verändert | 58.7s | 54.3s | **−4.4s** |
| L15 | stabil | 57.9s | 63.1s | **+5.2s** |
| L8 | stabil | 52.6s | 54.7s | +2.1s |
| L12 | stabil | 53.1s | 51.2s | −1.9s |

**Kernbefund:** Die veränderten Linien zeigen keine einheitliche Richtung — L11 stieg um +5.3s, L9 verbesserte sich um −4.4s. Noch deutlicher: L15 (stabil, keine GTFS-Änderung) erhöhte sich um +5.2s — praktisch identisch mit L11 (+5.3s). Da stabile und veränderte Linien die gleiche Bandbreite an Veränderungen zeigen, liegt die Quelle der Variation nicht im Netzwechsel selbst, sondern in externen Faktoren (saisonale Muster, Fahrgastzuwachs, Wetter).

**Starkes Finding:** Die VBZ hat den grössten Fahrplanwechsel in der Netzgeschichte ohne erkennbare Verspätungs-Disruption durchgeführt. Der Netzwechsel ist im Delay-Signal nicht sichtbar.

## Einlaufzeit — Performen neue Abschnitte anders?

In [ ]:
section_header("Einlaufzeit — neue vs. bestehende Haltestellen")
an.plot_einlaufzeit(changes, lf_all, cfg)

In [ ]:
log("Einlaufzeit Tabelle")
show_df(an.table_einlaufzeit(changes, lf_all))

**Beobachtung:** Die Einlaufzeit-Tabelle zeigt kein einheitliches Muster — das Ergebnis ist gemischt und linienabhängig.

**Neue vs. bestehende Haltestellen — Ø Delay ab Jan 2024:**
| Linie | Bestehende Halte (s) | Neue Halte (s) | Δ neu−best. |
|:---|---:|---:|---:|
| L11 (verändert) | 75.5 | 68.2 | **−7.3s** (neue besser) |
| L9 (verändert) | 57.4 | 50.8 | **−6.6s** (neue besser) |
| L13 (verändert) | 52.1 | 53.3 | +1.2s (≈ gleich) |
| L10 | 57.6 | 66.4 | **+8.8s** (neue schlechter) |
| L8 | 55.5 | 63.4 | **+7.9s** (neue schlechter) |
| L3 | 53.6 | 58.9 | +5.3s (neue schlechter) |
| L7 | 62.9 | 54.3 | −8.6s (neue besser) |
| L2 | 56.9 | 53.7 | −3.2s (neue besser) |

**Kernbefund für die "veränderten" Linien (L9, L11, L13):** Kein klassischer Einlaufeffekt. Im Gegenteil — L11 und L9 performen an neuen Haltestellen sogar besser als an bestehenden (−7.3s / −6.6s). Das spricht eher dafür, dass die "neuen" GTFS-Haltestellen keine problematischen Neubauabschnitte sind, sondern gut gelegene Innenstadthalte (→ konsistent mit der Stadtkreis-Analyse).

**Über alle Linien:** Kein eindeutiger Trend in eine Richtung — Einlaufzeit-Effekt ist nicht nachweisbar.

## Hotspots & Kaskaden — Kritische Knotenpunkte

In [ ]:
section_header("Hotspots & Kaskaden")
an.plot_hotspots(changes, lf_all, cfg)

In [ ]:
log("Hotspot Tabelle")
show_df(an.table_hotspots(changes, lf_all))

**Beobachtung:** Die Knotenpunkte mit den meisten Linien (Central: 7, Paradeplatz: 7) haben beide einen Ø Delay von ca. 49s — das liegt **unter** dem Gesamtdurchschnitt (~55s).

**Top-Hotspots nach Linienanzahl (j25):**
| Haltestelle | Linien | Ø Delay (s) | Beobachtungen |
|:---|---:|---:|---:|
| Central | 7 | 49.1 | 1 081 588 |
| Paradeplatz | 7 | 49.0 | 1 270 955 |
| Stauffacher | 6 | 60.2 | 986 075 |
| Bahnhofplatz/HB | 5 | 46.3 | 474 041 |
| Bürkliplatz | 5 | 52.3 | 917 620 |
| Bahnhofquai/HB | 5 | 53.4 | 880 893 |
| Milchbuck | 5 | 61.3 | 747 434 |

**Kernbefund:** Es gibt **keine positive Korrelation** zwischen Linienanzahl und Verspätung — das Gegenteil scheint möglich. Die grossen Knotenpunkte (Central, Paradeplatz, Bahnhofplatz) liegen alle unter dem Durchschnitt. Das Kaskadenrisiko-Modell "mehr Linien = mehr Delay" findet in den Daten keine Bestätigung. Höhere Delays entstehen offenbar nicht an den zentralen Umsteigeknoten, sondern anderswo im Netz (→ weiterführend in `03_analysis_4-spatial`).

## Versorgungsqualität — Welche Stadtteile profitieren?

In [ ]:
section_header("Versorgungsqualität nach Stadtkreis")
an.plot_service_quality_by_district(lf_all, cfg)
show_df(an.table_service_quality_by_district(lf_all))

**Beobachtung:** Der Chart zeigt die Veränderung der Linien-Anbindung pro Stadtkreis (Δ Anzahl verschiedener Linien, 2025 vs. 2023).

**Δ Linien pro Stadtkreis (j23 → j25):**
| Stadtkreis | j23 | j25 | Δ |
|:---|---:|---:|---:|
| Kreis 12 | 5 | 7 | **+2** |
| Kreis 4 | 11 | 13 | **+2** |
| Kreis 9 | 8 | 9 | +1 |
| Kreis 11 | 10 | 11 | +1 |
| Kreis 8 | 6 | 7 | +1 |
| Kreise 1, 2, 3, 5 | — | — | 0 |
| Kreis 10 | 4 | 3 | −1 |
| Kreis 6 | 14 | 13 | −1 |
| Kreis 7 | 11 | 9 | **−2** |

**Wichtiger Kontrast zu Stadtkreis-Chart oben:** Kreis 1 erhielt die meisten neuen Haltestellen (12), aber **null neue Linien** — die Innenstadt wird von denselben Linien bedient, die nun mehr Halte haben. Echte Anbindungs-Verbesserungen (neue Linien) liegen vor allem in Kreis 12 und Kreis 4.

**Verlierer:** Kreis 7 verliert 2 Linien (11→9). Kreis 6 und Kreis 10 verlieren je 1 Linie. Die Versorgungsqualität dieser Kreise hat sich im Betrachtungszeitraum verschlechtert.

## Fazit & Rückschluss auf die weitere Analyse

### Was die Netzanalyse für alle weiteren Notebooks bedeutet

Die GTFS-Analyse über 2023, 2024 und 2025 ergibt folgende strukturelle Erkenntnisse:

#### Verändertes Teilnetz — Jahresvergleich mit Vorbehalt
| Linie | j23 | j24 | j25 | Kontext |
| :---: | ---: | ---: | ---: | :--- |
| **9** | 24 Halte | 32 Halte | 32 Halte | +8 neue Abschnitte ab Dez 2023 |
| **11** | 20 Halte | 33 Halte | 34 Halte | +13 neue Abschnitte ab Dez 2023 |
| **13** | 11 Halte | 30 Halte | 30 Halte | +19 Halte (+173%) ab Dez 2023 |

Für diese Linien ist Linie 9 in 2023 strukturell eine **andere** Linie als Linie 9 in 2024–2025.

> **⚠️ Externe Recherche-Korrektur (Perplexity, Mai 2026):**
> Die VBZ-Medienmitteilung zum Fahrplanwechsel Dez 2023 nennt **keine formalen Streckenumbauten** für Tramlinien 9, 11, 13 — die ausgewiesenen Änderungen betrafen primär Buslinien. Die GTFS-Unterschiede könnten daher sein:
> - **Flexity-Rollout**: neue Fahrzeuge auf Linien 11/13 → präzisere Haltestellenaufzeichnung im GTFS
> - **Takt-/Umlaufänderungen**: mehr Kurse, andere Wendeäste → neue Stop-IDs im GTFS
> - **GTFS-Modellierungsartefakt**: Fahrplankopplungen die als neue Halte erscheinen
>
> Das bedeutet: Die "+173%" bei Linie 13 sind real im GTFS-Datensatz sichtbar, aber die Ursache ist unklar. Das `gtfs_year`-Feature ist trotzdem valide — es kodiert einen echten Zeitschnitt. Ob es Netzstruktur oder nur einen Zeiteffekt erfasst, zeigt der Modellvergleich.

#### Stabiles Referenznetz
Linien mit identischer Stoppanzahl j23 = j24 = j25 können jahresübergreifend direkt verglichen werden. *Welche Linien das genau sind zeigt das Quantifizierungs-Chart nach Ausführung.*

#### Das `gtfs_year`-Feature — kritische Bewertung

```python
# In 02_preparation.ipynb hinzufügen:
pl.when(pl.col("operating_date") < pl.lit("2024-01-01").str.to_date())
  .then(pl.lit("j23"))
  .otherwise(pl.lit("j24_j25"))
  .alias("gtfs_year")
```

**Was es codiert:** Den Zeitschnitt Dez 2023 — für Linien 9, 11, 13 gibt es im GTFS einen markanten Unterschied. Ob das eine echte Streckenänderung oder ein Modellierungsartefakt ist, ist offen.

**Limitierung:** Für strukturell stabile Linien ist `gtfs_year` eine reine Zeitvariable ohne Netz-Kontext. Ob es tatsächlich die Modellleistung verbessert, **muss empirisch getestet werden**.

**Alternative:** `n_stops_line` als kontinuierliches Signal — trifft den gleichen Sachverhalt, ohne die binäre Vereinfachung.

**Entscheidung:** Feature als Kandidat aufnehmen, im Modellvergleich evaluieren.

#### Hinweis für alle Analysis-Notebooks
> Alle Analysen in `03_analysis_4-spatial`, `03_analysis_3-temporal`, `03_analysis_5-meteo` und `03_analysis_6-events`
> sollten bei Linien-bezogenen Befunden den Netzwechsel Dezember 2023 als Kontextinformation nennen.
> Detaillierte Aufschlüsselung immer mit Verweis auf dieses Notebook: `03_analysis_2-network.ipynb`.

---

#### 🔜 Offenes TODO: Kaskadenanalyse mit `trip_id` (F-NET-07)

Eine wichtige Folgefrage aus der Target-Analyse: **Wenn ein Trip mit Verspätung endet — startet der nächste Trip (selbes Fahrzeug, andere Richtung) dann ebenfalls zu spät?**

> **Wie die Analyse funktioniert:**
> VBZ plant an den Endpunkten Wendezeit ein (typisch 5–10 Min). Bei moderaten Verspätungen wird diese Wendezeit aufgebraucht und der nächste Trip startet pünktlich. Bei Extremverspätungen (> Wendezeit) kann die Verspätung auf den nächsten Trip übertragen werden — das ist der Kaskadeneffekt.
>
> Mit `trip_id` und der Sortierung nach `operating_date` + `stop_sequence` lässt sich für jeden Trip der letzte Stop-Delay extrahieren und mit dem ersten Stop-Delay des Nachfolge-Trips vergleichen.

**Warum das für die Modellierung wichtig ist:**
Ein Feature `prev_trip_end_delay` (Verspätung am Ende des vorherigen Trips) wäre ein starkes Vorhersage-Signal — insbesondere in den Abendstunden wenn sich Verspätungen aufschaukeln. Das könnte den 21h-Peak aus F-TEMP-01 teilweise erklären.

```python
# Skizze für 02_preparation oder Modellierungsphase:
# trip_end_delay = lf.group_by("trip_id").agg(
#     pl.col("arrival_delay").last().alias("trip_end_delay")
# )
# → join mit nächstem Trip über Fahrzeug-ID / Umlauf-ID
```

**Status:** Offen — Daten sind vorhanden (`trip_id` im Master-Set), Implementierung ausstehend. → F-NET-07


## Key Findings

→ Vollständige Findings-Tabelle mit Impact und Action in [`03_analysis_0-overview.ipynb`](03_analysis_0-overview.ipynb).

| ID | Finding | Status |
|:---|:---|:---|
| F-NET-01 | Im GTFS zeigen L9/L11/L13 zum Fahrplanwechsel Dez 2023 markante Haltestellen-Zunahmen (+8/+13/+19). Keine formalen Tramstrecken-Umbauten dokumentiert — wahrscheinlich Flexity-Rollout oder Taktänderungen. Die neuen Halte konzentrieren sich auf die Innenstadt-Kreise (Kreis 1 top mit 12), nicht auf die erwartete Peripherie. | done |
| F-NET-02 | Stabile Referenzlinien (L10, L12, L14, L17): identische Haltestellen-Zahl über alle drei Jahre — direkte Jahresvergleiche möglich. Anomalie L2: 31→21→31 Halte (j24-Dip), wahrscheinlich GTFS-Routing-Variante. | done |
| F-NET-03 | `gtfs_year` Feature (`j23` vs `j24_j25`) kodiert den Zeitschnitt Dez 2023. Ob es Netzstruktur oder Zeiteffekt misst, zeigt der Modellvergleich. | done |
| F-NET-04 | Kein Einlaufzeit-Effekt nachweisbar: L11 und L9 performen an neuen GTFS-Haltestellen sogar besser als an bestehenden (−7.3s / −6.6s). Kein einheitlicher Trend über alle Linien. | done |
| F-NET-05 | Keine positive Korrelation zwischen Linienanzahl und Delay: Knotenpunkte Central und Paradeplatz (je 7 Linien) liegen bei 49s — unter dem Netz-Durchschnitt (~55s). Das Kaskadenrisiko-Modell findet in den Daten keine Bestätigung. | done |
| F-NET-06 | Versorgungsqualität (Δ Linien): Kreis 12 (+2) und Kreis 4 (+2) gewinnen am meisten. Kreis 7 verliert 2 Linien. Kreis 1 erhielt die meisten neuen Haltestellen (12) aber keine neuen Linien. | done |
| F-NET-07 | `trip_id` ermöglicht Kaskadenanalyse: Verspätungsübertragung von Fahrt zu Fahrt messbar — in `03_analysis_2-network` vertiefen. | done |
| F-NET-08 | **Linie E** ist eine Entlastungs-/Verstärkerlinie (planmässig im GTFS modelliert) mit 128.1s Ø Delay — klarer Ausreisser. Kein Datenfehler; im Modell behalten, aber als Sonderlinie annotieren. | done |